In [1]:
import json
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from benchmark.config import (
    DATA_PROCESSED,
    DATASETS,
    MODELS_DIR,
    N_TRIALS,
    RESULTS_BENCHMARK,
)
from benchmark.models.random_forest_model import fit_random_forest_cross_validation
from benchmark.models.random_forest_model import fit_best_random_forest
from sklearn.preprocessing import LabelEncoder

/home/shado/college_project/comp9417/9417GroupProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_name = "sleep_health"
cfg = DATASETS[data_name]
processed = DATA_PROCESSED / data_name

train_df = pd.read_csv(processed / "train.csv")
test_df = pd.read_csv(processed / "test.csv")

with open(processed / "metadata.json", "r") as f:
    metadata = json.load(f)
print(metadata)

{'num_cols': ['age', 'bmi', 'sleep_duration_hrs', 'sleep_quality_score', 'rem_percentage', 'deep_sleep_percentage', 'sleep_latency_mins', 'wake_episodes_per_night', 'caffeine_mg_before_bed', 'alcohol_units_before_bed', 'screen_time_before_bed_mins', 'exercise_day', 'steps_that_day', 'nap_duration_mins', 'stress_score', 'work_hours_that_day', 'heart_rate_resting_bpm', 'room_temperature_celsius', 'weekend_sleep_diff_hrs', 'rem_hrs', 'deep_sleep_hrs', 'light_sleep_hrs', 'wake_rate'], 'cat_cols': ['gender', 'occupation', 'country', 'chronotype', 'mental_health_condition', 'season', 'day_type'], 'bin_cols': ['sleep_aid_used', 'shift_work'], 'target_col': 'sleep_disorder_risk'}


In [3]:
target_column = metadata["target_col"]

x_train = train_df.drop(columns=[target_column]).copy()
x_test = test_df.drop(columns=[target_column]).copy()

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df[target_column])
y_test = label_encoder.transform(test_df[target_column])

In [4]:
num_cols = metadata["num_cols"]
cat_cols = metadata["cat_cols"]
bin_cols = metadata["bin_cols"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
        ("bin", OrdinalEncoder(), bin_cols),
    ],
    remainder="passthrough",
)

In [5]:
best_trial = fit_random_forest_cross_validation(
    X=x_train,
    y=y_train,
    processor=preprocessor,
    task=cfg["task"],
    n_trials=N_TRIALS,
)

rf_model, rf_pipeline, rf_fit_time = fit_best_random_forest(
    X=x_train,
    y=y_train,
    best_trial=best_trial,
    processor=preprocessor,
    task=cfg["task"],
)

[I 2026-04-20 19:12:07,836] A new study created in memory with name: no-name-bc9cf0ba-e9fd-4240-ba70-c9925db1b617
[I 2026-04-20 19:12:08,444] Trial 0 finished with value: 0.611788943601927 and parameters: {'n_estimators': 32, 'max_depth': 7, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.611788943601927.
[I 2026-04-20 19:12:09,364] Trial 1 finished with value: 0.5856359459529312 and parameters: {'n_estimators': 67, 'max_depth': 8, 'min_samples_split': 11, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': True}. Best is trial 0 with value: 0.611788943601927.
[I 2026-04-20 19:12:10,658] Trial 2 finished with value: 0.6523721894919294 and parameters: {'n_estimators': 112, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': True}. Best is trial 2 with value: 0.6523721894919294.
[I 2026-04-20 19:12:11,684] Trial 3 finished with value: 0.672064288002133 and par

In [6]:
import time
from benchmark.metrics import evaluate_classification

x_test_transformed = rf_pipeline.transform(x_test)

t0 = time.perf_counter()
y_test_proba = rf_model.predict_proba(x_test_transformed)
inference_time = time.perf_counter() - t0

inference_time_per_sample = inference_time / len(x_test)

test_metrics = evaluate_classification(y_test, y_test_proba)


print("Best params:", best_trial.params)
print("Training time (s):", rf_fit_time)
print("Inference time per sample (s):", inference_time_per_sample)
print("Test metrics:", test_metrics)

Best params: {'n_estimators': 82, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'bootstrap': True}
Training time (s): 0.14770669900008215
Inference time per sample (s): 6.562166749972675e-06
Test metrics: {'accuracy': 0.8458, 'auc_roc': 0.9682}


In [7]:
models_dir = MODELS_DIR / data_name
models_dir.mkdir(parents=True, exist_ok=True)

results_dir = RESULTS_BENCHMARK / data_name
results_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_model, models_dir / "random_forest.joblib")
joblib.dump(rf_pipeline, models_dir / "random_forest_pipeline.joblib")

output = {
    **test_metrics,
    "fit_time_s": float(rf_fit_time),
    "infer_time_per_sample_s": float(inference_time_per_sample),
    "best_params": best_trial.params,
}

with open(results_dir / "random_forest.json", "w") as f:
    json.dump(output, f, indent=4)

print(results_dir / "random_forest.json")

/home/shado/college_project/comp9417/9417GroupProject/results/sleep_health/random_forest.json


In [8]:
from benchmark.metrics import plot_roc_curve
from benchmark.metrics import plot_confusion_matrix
from benchmark.metrics import save_classification_report

results_dir = RESULTS_BENCHMARK / data_name
results_dir.mkdir(parents=True, exist_ok=True)

# ===== ROC Curve =====
plot_roc_curve(
    y_test,
    y_test_proba,
    save_path=results_dir / "roc_curve_random_forest.png",
    class_names=list(label_encoder.classes_),
)

# ===== Confusion Matrix =====
plot_confusion_matrix(
    y_test,
    y_test_proba,
    save_path=results_dir / "confusion_matrix_random_forest.png",
    class_names=list(label_encoder.classes_),
)

# ===== Classification Report =====
save_classification_report(
    y_test,
    y_test_proba,
    save_path=results_dir / "classification_report_random_forest.json",
    class_names=list(label_encoder.classes_),
)

print("Saved:")
print(results_dir / "roc_curve_random_forest.png")
print(results_dir / "confusion_matrix_random_forest.png")
print(results_dir / "classification_report_random_forest.json")

Saved:
/home/shado/college_project/comp9417/9417GroupProject/results/sleep_health/roc_curve_random_forest.png
/home/shado/college_project/comp9417/9417GroupProject/results/sleep_health/confusion_matrix_random_forest.png
/home/shado/college_project/comp9417/9417GroupProject/results/sleep_health/classification_report_random_forest.json
